# Grammy 2027: compact Bayesian nomination models
**Stage 1: equal-probability baseline and eight independent Bayesian probit models.**

Team: Yegor Lazarev, Alexander Shayko, Diana Shilova.

This revision aggregates 71 award columns into 15 predictors and combines them with the original 12 chart/Grammy predictors: **27 predictors per category**. It evaluates four prespecified test seasons (2022–2025), requiring **32 backtest fits plus eight final fits**. Each fit uses two chains, 300 warmup iterations and 500 retained draws per chain. The earlier 83-predictor model's scores do not apply to this revision.

NUTS is initialized at the training posterior mode and uses its local posterior covariance as a fixed mass-matrix preconditioner. This numerical optimization does not replace the posterior with a Gaussian approximation: the final probabilities still integrate NUTS draws from the specified probit posterior. Priors and likelihood are evaluated exactly as stated below, subject to numerical precision and MCMC error.

**First run:** execute through the **single-fit pilot** and inspect its diagnostics and time before starting the backtest. Kaggle input and output paths are detected below. With two visible GPUs, the two chains run in parallel. Completed fits are cached. No full training/backtest results for this revision are supplied; local validation checks, a small synthetic sampler check and a short real-data sampler check are described in the accompanying README.

The target is artist-level association with a nomination in each category, with probabilities and top-five artist lists. Forecasting winning works and artist award-count distributions remains outside this Stage-1 notebook.

## 1. Environment
On Kaggle, retain the working GPU-enabled JAX installation. Install NumPyro only if needed; restart the session after changing dependencies. Do not install a CPU-only JAX package over a working GPU environment. Actual package versions and devices are recorded below.

In [ ]:
# Optional, only if dependencies are missing:
# %pip install numpyro pandas scipy matplotlib

In [ ]:
import os
# These defaults limit CPU oversubscription; set before importing numerical libraries.
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")

import gc, gzip, hashlib, json, time, warnings
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import ndtr, ndtri, log_ndtr as scipy_log_ndtr
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve
from scipy.stats import t as student_t
from IPython.display import display
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import log_ndtr
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_value
from numpyro.diagnostics import summary as mcmc_summary

VERSIONS = {p: metadata.version(p) for p in
            ["numpy", "pandas", "scipy", "matplotlib", "jax", "jaxlib", "numpyro"]}
print(VERSIONS)
print("Devices:", jax.devices())

## 2. Forecast protocol and computational budget

- Season `t` uses the source's approximate October–September chart window and corresponds to the ceremony in `t+1`.
- Backtest seasons are **2022, 2023, 2024 and 2025**. Each training set contains only earlier seasons. These four recent seasons are fixed for computational reasons, not selected according to forecast accuracy.
- All eight categories are retained, giving 32 held-out category-season events. Final forecasts for season 2026 (the 2027 ceremony) add eight fits.
- Relative to the former 72-fit/750+750 plan, the default iteration budget is `40 * 2 * (300+500)` rather than `72 * 2 * (750+750)`, about 30% as many outer iterations. Runtime is not proportional to that number: trajectory lengths, compilation and model geometry also matter.
- Retained draws and warmup are reduced only with the existing diagnostic gates in force. If a fit fails, increase its budget or revise the computation and re-run; do not suppress failed diagnostics to save time.
- The four seasons provide fewer independent comparison occasions than the previous eight-season plan. Statistical model comparisons are exploratory and have limited power.

The pool and labels are unchanged: charting artists, with performer-based attribution of nominated works. Genre and formal eligibility are not verified. The actual forecast is made before the nomination announcement using the supplied information alignment. Calendar dates in the original proposal have not been independently re-audited here.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input/datasets/shaykoaleksandr/grammy-dataset/Grammy Nomination Dataset.csv")
INPUT_PATH = KAGGLE_INPUT if KAGGLE_INPUT.is_file() else None
FORECAST_SEASON = 2026
TEST_SEASONS = [2022, 2023, 2024, 2025]
TOP_K = 5
SEED = 20261021
NUM_WARMUP = 300
NUM_SAMPLES = 500
NUM_CHAINS = 2
TARGET_ACCEPT = 0.90
MAX_TREE_DEPTH = 10
ALPHA_LOCATION = float(ndtri(0.01))
ALPHA_SCALE = 1.0
BETA_SCALE = 1.0
MAX_RHAT = 1.05
MIN_ESS = 100
MAP_GRAD_TOLERANCE = 1e-4
MODEL_VERSION = "compact27-map-metric-v3"
PRECONDITIONER = "fixed inverse posterior Hessian at training MAP"
CHAIN_METHOD = "parallel" if jax.default_backend() == "gpu" and jax.local_device_count() >= NUM_CHAINS else "sequential"

CATEGORIES = ["album_of_the_year", "record_of_the_year", "song_of_the_year",
              "best_new_artist", "pop_vocal_album", "pop_solo",
              "pop_duo_group", "music_video"]

# Source category stems used to define relevant award recognition.
# Empty lists mean no direct counterpart; the corresponding feature is zero.
RELEVANT_AWARD_STEMS = {
    "ama": {
        "album_of_the_year": ["album_of_the_year", "pop_album"],
        "record_of_the_year": ["song_of_the_year", "pop_song", "collaboration_of_the_year"],
        "song_of_the_year": ["song_of_the_year", "pop_song", "collaboration_of_the_year"],
        "best_new_artist": ["new_artist_of_the_year"],
        "pop_vocal_album": ["pop_album", "album_of_the_year"],
        "pop_solo": ["pop_song", "pop_male_artist", "pop_female_artist"],
        "pop_duo_group": ["pop_duo_group", "collaboration_of_the_year"],
        "music_video": ["music_video"],
    },
    "vma": {
        "album_of_the_year": [],
        "record_of_the_year": ["song_of_the_year", "best_pop", "best_collaboration"],
        "song_of_the_year": ["song_of_the_year", "best_pop", "best_collaboration"],
        "best_new_artist": ["best_new_artist"],
        "pop_vocal_album": ["best_pop"],
        "pop_solo": ["best_pop", "best_male_video", "best_female_video"],
        "pop_duo_group": ["best_group", "best_collaboration"],
        "music_video": ["video_of_the_year"],
    },
    "iheart": {
        "album_of_the_year": [],
        "record_of_the_year": ["song_of_the_year", "pop_song_of_the_year", "best_collaboration"],
        "song_of_the_year": ["song_of_the_year", "pop_song_of_the_year", "best_collaboration"],
        "best_new_artist": ["best_new_artist", "best_new_pop_artist"],
        "pop_vocal_album": ["pop_artist_of_the_year"],
        "pop_solo": ["pop_song_of_the_year", "pop_artist_of_the_year", "male_artist_of_the_year", "female_artist_of_the_year"],
        "pop_duo_group": ["duo_group_of_the_year", "best_collaboration"],
        "music_video": ["best_music_video"],
    },
}

OUTPUT = Path("/kaggle/working/results_stage1_fast") if Path("/kaggle/working").is_dir() else Path("results_stage1_fast")
CACHE = OUTPUT / "posterior_cache"
CACHE.mkdir(parents=True, exist_ok=True)
if NUM_CHAINS < 2:
    raise ValueError("Use at least two chains for convergence checks.")
if not TEST_SEASONS or max(TEST_SEASONS) >= FORECAST_SEASON:
    raise ValueError("Test seasons must precede the forecast season.")
print("Chain execution:", CHAIN_METHOD, "| planned fits:", len(CATEGORIES)*(len(TEST_SEASONS)+1))
print("Output:", OUTPUT.resolve())

## 3. Input and integrity checks

The supplied dataset was built by `grammy_nomination_dataset.ipynb`. Its inputs include Billboard chart snapshots, the Grammy label/performer files, and the prepared AMA, VMA and iHeartRadio tables supplied with the project. Their seasonal alignment is retained without another lag. The preparation notebook and raw inputs are preserved in `source_project/` in the archive for provenance; this model notebook does not need to download data.

The SHA-256 below identifies the **decompressed CSV bytes** used for this run. If the dataset is intentionally revised, audit the revision and update the expected hash explicitly. Final-season labels must remain missing. `won` is never a predictor.

References: [Billboard source archive](https://github.com/utdata/rwd-billboard-data/tree/f5801b03d5e629845e65213ff236df98f1aae3fd), [Grammy awards archive](https://www.grammy.com/awards), [NumPyro documentation](https://num.pyro.ai/en/stable/). The exact provenance and extraction methods for the supplemental label files still need verification before the final course submission.

In [ ]:
EXPECTED_SHA256 = "f2f8a3af293485500c7a792ed333bffaf7671aa075ef49f4f193633bb9d6b62c"
candidates = ([Path(INPUT_PATH)] if INPUT_PATH is not None else [
    Path("data/grammy_nomination_dataset.csv.gz"),
    Path("data/grammy_nomination_dataset.csv"),
    Path("Grammy Nomination Dataset.csv"),
    Path("grammy_nomination_dataset.csv"),
])
DATA_PATH = next((p for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Extract the archive or set INPUT_PATH to the supplied CSV.")
opener = gzip.open if DATA_PATH.suffix == ".gz" else open
with opener(DATA_PATH, "rb") as f:
    DATA_SHA256 = hashlib.file_digest(f, "sha256").hexdigest()
if DATA_SHA256 != EXPECTED_SHA256:
    raise ValueError("Input differs from the audited dataset. Review it before changing EXPECTED_SHA256.")
data = pd.read_csv(DATA_PATH)
INPUT_SHAPE = data.shape
AWARD_TOTAL_COLUMNS = ["ama_noms_total", "ama_wins_prev_total",
                       "vma_noms_total", "vma_wins_total",
                       "iheart_noms_total", "iheart_wins_total"]
AWARD_HELD_COLUMNS = ["ama_held", "vma_held", "iheart_held"]
AWARD_FLAG_SUFFIXES = ("_ama_nominated", "_ama_winner_prev",
                       "_vma_nominated", "_vma_winner",
                       "_iheart_nominated", "_iheart_winner")
AWARD_FLAG_COLUMNS = sorted(c for c in data.columns if c.endswith(AWARD_FLAG_SUFFIXES))
AWARD_COLUMNS = AWARD_FLAG_COLUMNS + AWARD_TOTAL_COLUMNS + AWARD_HELD_COLUMNS
if len(AWARD_FLAG_COLUMNS) != 62 or len(set(AWARD_COLUMNS)) != 71:
    raise ValueError("Expected 62 award-category flags, six totals and three ceremony indicators.")
missing_award_columns = sorted(set(AWARD_COLUMNS) - set(data.columns))
if missing_award_columns:
    raise ValueError(f"Missing prepared award features: {missing_award_columns}")
MODEL_COLUMNS = ["season", "cat", "artist_key", "artist", "nominated",
                 "h_best_peak", "b_best_peak", "h_entries", "b_entries",
                 "h_weeks", "b_weeks", "h_top10_weeks", "b_top10_weeks",
                 "prior_noms", "prior_wins", "prior_noms_same_cat", "new_in_window"] + AWARD_COLUMNS
data = data[MODEL_COLUMNS].copy()  # retain only model columns to reduce memory usage
keys = ["season", "cat", "artist_key"]
if data[keys].isna().any().any() or data.duplicated(keys).any():
    raise ValueError("Missing or duplicated artist-season-category keys.")
if set(data.cat) != set(CATEGORIES):
    raise ValueError("Unexpected categories.")
data = data.sort_values(keys).reset_index(drop=True)
historical = data.loc[data.season < FORECAST_SEASON].copy()
future = data.loc[data.season == FORECAST_SEASON].copy()
if historical.nominated.isna().any() or not historical.nominated.isin([0, 1]).all():
    raise ValueError("Historical nomination labels must be binary and complete.")
if future.empty or future.nominated.notna().any():
    raise ValueError("Forecast-season outcomes must be unavailable.")
if not set(TEST_SEASONS).issubset(set(historical.season)):
    raise ValueError("A requested test season is absent.")
print("Input:", DATA_PATH, "| source shape:", INPUT_SHAPE, "| SHA-256:", DATA_SHA256)
display(historical.groupby("cat").agg(
    observations=("nominated", "size"), nominated_artists=("nominated", "sum"),
    seasons=("season", "nunique"), nomination_rate=("nominated", "mean")))
display(data.groupby("season").agg(rows=("artist_key", "size"),
                                  observed_labels=("nominated", "count")))

## 4. Aggregating other-award information

The original 12 chart/Grammy predictors are retained: chart peak strength, log entries/weeks/top-10 weeks for each chart, log prior Grammy nominations/wins/category nominations, and the existing new-in-window flag.

The other 71 columns are replaced by **15 predictors**:

| Group | Count | Construction |
|---|---:|---|
| Overall recognition at each award | 6 | Existing total nominations and wins, transformed with `log1p` |
| Ceremony presence | 3 | Existing `ama_held`, `vma_held`, `iheart_held` |
| Recognition relevant to the target category | 6 | Any nomination and any win in the source categories listed in `RELEVANT_AWARD_STEMS`, for each award |

For example, Best New Artist uses new-artist recognition, Music Video uses music-video recognition, and Pop Duo/Group uses group/collaboration recognition. For Album of the Year, VMA/iHeart have no direct album counterpart in this input: their relevant-category flags are zero, while their overall recognition totals remain available. For Pop Vocal Album, pop-performance/artist awards are broad proxies, not equivalent album awards. The mapping is a prespecified modelling judgement, not an official equivalence of award categories.

An `any` indicator uses a maximum, rather than summing overlapping category flags. The global totals describe broad recognition; the relevant flags preserve some target-specific information that totals alone would discard. Aggregation intentionally discards some fine detail; whether this helps prediction must be evaluated by the backtest. This revision is not statistically identical to the former 83-predictor specification.

**Timing is preserved:** current mapped-season nominations for all three awards, current VMA/iHeart winners, and already-lagged most-recent-earlier-ceremony AMA winners. There is no additional annual shift. Current `ama_held` does not erase earlier AMA wins. The supplied availability convention is adopted; historical publication dates have not been independently re-audited.

Every predictor is standardized using training observations only, separately for each fold/category. Training-constant columns get scale one. The model uses 27 coefficients plus an intercept. Normal priors regularize remaining dependence between totals and relevant-category flags. No class weighting or negative-class subsampling is introduced.

In [ ]:
def make_features(frame):
    out = pd.DataFrame(index=frame.index)
    for prefix, size in [("h", 100), ("b", 200)]:
        peak = frame[f"{prefix}_best_peak"]
        if not peak.dropna().between(1, size).all():
            raise ValueError(f"Invalid {prefix} chart peak.")
        out[f"{prefix}_peak_strength"] = ((size + 1 - peak) / size).fillna(0)
        for field in ["entries", "weeks", "top10_weeks"]:
            name = f"{prefix}_{field}"
            if frame[name].isna().any() or (frame[name] < 0).any():
                raise ValueError(f"Invalid chart counts: {name}")
            out[f"log_{name}"] = np.log1p(frame[name])
    for name in ["prior_noms", "prior_wins", "prior_noms_same_cat"]:
        if frame[name].isna().any() or (frame[name] < 0).any():
            raise ValueError(f"Invalid Grammy history: {name}")
        out[f"log_{name}"] = np.log1p(frame[name])
    if not frame.new_in_window.isin([0, 1]).all():
        raise ValueError("new_in_window must be binary.")
    out["new_in_window"] = frame.new_in_window.astype(float)

    # Check all source columns even though they are aggregated below.
    for name in AWARD_FLAG_COLUMNS + AWARD_HELD_COLUMNS:
        if not frame[name].isin([0, 1]).all():
            raise ValueError(f"Invalid award indicator: {name}")
    for name in AWARD_TOTAL_COLUMNS:
        values = frame[name]
        if values.isna().any() or (values < 0).any() or not np.equal(values, np.floor(values)).all():
            raise ValueError(f"Invalid award count: {name}")
        out[f"log_{name}"] = np.log1p(values)
    for name in AWARD_HELD_COLUMNS:
        out[name] = frame[name].astype(float)

    for award, mapping in RELEVANT_AWARD_STEMS.items():
        if set(mapping) != set(CATEGORIES):
            raise ValueError(f"Incomplete category mapping: {award}")
        for kind, source_suffix in [("nominated", "nominated"),
                                    ("winner", "winner_prev" if award == "ama" else "winner")]:
            feature = f"{award}_relevant_{kind}"
            out[feature] = 0.0
            for cat, stems in mapping.items():
                columns = [f"{stem}_{award}_{source_suffix}" for stem in stems]
                if columns:
                    mask = frame.cat.eq(cat)
                    out.loc[mask, feature] = frame.loc[mask, columns].max(axis=1).astype(float)
    if out.shape[1] != 27 or not np.isfinite(out.to_numpy()).all():
        raise ValueError("Expected 27 finite predictors.")
    return out

FEATURE_NAMES = make_features(data.head(10)).columns.tolist()

def training_scaler(frame):
    raw = make_features(frame).to_numpy(dtype=float)
    center, scale = raw.mean(axis=0), raw.std(axis=0, ddof=0)
    return center, np.where(scale < 1e-12, 1.0, scale)

def transform(frame, center, scale):
    return (make_features(frame).to_numpy(dtype=float) - center) / scale

print("Predictors:", len(FEATURE_NAMES), FEATURE_NAMES)
assert not {"won", "nominated", "season"}.intersection(FEATURE_NAMES)
display(pd.DataFrame([{"award": award, "target_category": cat,
    "source_category_stems": ", ".join(stems) or "No direct counterpart"}
    for award, mapping in RELEVANT_AWARD_STEMS.items() for cat, stems in mapping.items()]))

## 5. Competing forecasts

**Equal-probability baseline.** Within a category, every test candidate receives the training nomination rate,
\[
\hat p_{0,c}=\frac{\sum_{i\in\mathrm{train},c} y_i}{n_{\mathrm{train},c}}.
\]
This uses neither current labels nor the current number of nominees. Five artists do not equal five nomination slots: a collaboration can nominate several artists, and one artist can have several nominated works. Consequently, we do **not** set the baseline to `5 / N` and do **not** force probit probabilities to sum to five.

**Eight independent Bayesian probit regressions.** For each category \(c\):
\[
y_i\mid\alpha_c,\beta_c\sim\mathrm{Bernoulli}\{\Phi(\alpha_c+x_i^\top\beta_c)\},\qquad
\alpha_c\sim N\{\Phi^{-1}(0.01),1\},\quad \beta_{cj}\sim N(0,1).
\]
The intercept prior is centered at a rare-event probability of 1% for an average standardized candidate. The priors are fixed across folds; coefficients are not shared across categories. The reported probability is the posterior average of \(\Phi(\alpha_c+x_i^\top\beta_c)\), not the probit of the posterior-average coefficients.

We evaluate the Bernoulli log likelihood with stable log-normal-CDF functions. NUTS diagnostics are exported for every fit. A fit is blocked if it has divergences, split R-hat above 1.05, effective sample size below 100, or reaches the tree-depth limit. These are minimum screening thresholds, not proof that the model is correct. For a final report, inspect fits near these limits; R-hat near 1.01 is preferable.

**Efficient initialization and geometry.** On each training fold, optimize the same log posterior to find its mode (MAP). Its negative Hessian is a local posterior precision; its inverse is supplied to NumPyro as the fixed `inverse_mass_matrix`. This convention matches NumPyro's mass-matrix covariance adaptation. The metric controls proposal geometry and the MAP controls initialization; neither changes the prior or likelihood. Mass-matrix adaptation is disabled to retain this training-based preconditioner, while step-size adaptation remains enabled during warmup. A poor local metric may still mix inefficiently; all sampling diagnostics remain mandatory.

This addresses coefficient dependence without increasing the tree-depth cap or accepting failed convergence. It is separate from feature aggregation: the metric preserves the compact model's posterior, whereas aggregation changes its predictor specification.

References: [NumPyro NUTS API](https://num.pyro.ai/en/stable/mcmc.html), [NumPyro mass-matrix implementation](https://num.pyro.ai/en/stable/_modules/numpyro/infer/hmc_util.html), [Stan on correlated predictors and HMC](https://mc-stan.org/learn-stan/case-studies/qr_regression.html).

In [ ]:
def prior_parameters(n_features):
    location = np.r_[ALPHA_LOCATION, np.zeros(n_features)]
    scale = np.r_[ALPHA_SCALE, np.full(n_features, BETA_SCALE)]
    return location, scale

def posterior_objective(coef, Z, y, prior_location, prior_scale):
    signed_eta = (2*y-1) * (Z @ coef)
    log_cdf = scipy_log_ndtr(signed_eta)
    inverse_mills = np.exp(-0.5*signed_eta**2 - 0.5*np.log(2*np.pi) - log_cdf)
    precision = 1 / prior_scale**2
    delta = coef - prior_location
    value = -log_cdf.sum() + 0.5*np.sum(precision*delta**2)
    gradient = -Z.T @ ((2*y-1)*inverse_mills) + precision*delta
    # Roundoff can make an otherwise nonnegative curvature slightly negative.
    curvature = np.maximum(inverse_mills*(inverse_mills+signed_eta), 0.0)
    hessian = Z.T @ (curvature[:, None]*Z) + np.diag(precision)
    return value, gradient, hessian

def initialize_geometry(X, y):
    Z = np.column_stack([np.ones(len(X)), X])
    location, scale = prior_parameters(X.shape[1])
    start = location.copy()
    start[0] = float(ndtri(np.clip(y.mean(), 1e-6, 1-1e-6)))
    args = (Z, y, location, scale)
    result = minimize(lambda w: posterior_objective(w, *args)[0], start,
        jac=lambda w: posterior_objective(w, *args)[1],
        hess=lambda w: posterior_objective(w, *args)[2],
        method="trust-exact", options={"gtol": 1e-6, "maxiter": 100})
    value, gradient, precision = posterior_objective(result.x, *args)
    max_gradient = float(np.max(np.abs(gradient)))
    if not np.isfinite(value) or not np.isfinite(gradient).all() or max_gradient > MAP_GRAD_TOLERANCE:
        raise RuntimeError(f"Posterior-mode optimization failed: {result.message}; gradient={max_gradient}")
    covariance = cho_solve(cho_factor(precision, lower=True), np.eye(len(result.x)))
    covariance = (covariance + covariance.T) / 2
    np.linalg.cholesky(covariance)
    return result.x, covariance, max_gradient

def probit_model(X, y):
    location = jnp.concatenate([jnp.asarray([ALPHA_LOCATION]), jnp.zeros(X.shape[1])])
    scale = jnp.concatenate([jnp.asarray([ALPHA_SCALE]), jnp.full((X.shape[1],), BETA_SCALE)])
    coef = numpyro.sample("coef", dist.Normal(location, scale).to_event(1))
    eta = coef[0] + X @ coef[1:]
    numpyro.factor("observations", jnp.where(y == 1, log_ndtr(eta), log_ndtr(-eta)).sum())

def fit_probit(train, category, predict_season):
    if train.empty or train.season.max() >= predict_season:
        raise ValueError("Training must strictly precede prediction.")
    if train.cat.nunique() != 1 or train.cat.iloc[0] != category:
        raise ValueError("Each fit must contain one category.")
    center, scale = training_scaler(train)
    signature = dict(model=MODEL_VERSION, data=DATA_SHA256, category=category,
        predict_season=int(predict_season), features=FEATURE_NAMES, mapping=RELEVANT_AWARD_STEMS,
        seed=SEED, warmup=NUM_WARMUP, samples=NUM_SAMPLES, chains=NUM_CHAINS,
        chain_method=CHAIN_METHOD, backend=jax.default_backend(), preconditioner=PRECONDITIONER,
        target_accept=TARGET_ACCEPT, max_depth=MAX_TREE_DEPTH,
        map_gradient_tolerance=MAP_GRAD_TOLERANCE,
        alpha_location=ALPHA_LOCATION, alpha_scale=ALPHA_SCALE, beta_scale=BETA_SCALE,
        versions=VERSIONS)
    digest = hashlib.sha256(json.dumps(signature, sort_keys=True).encode()).hexdigest()[:16]
    cache_file = CACHE / f"{predict_season}_{category}_{digest}.npz"
    diag_file = cache_file.with_suffix(".json")
    if cache_file.exists() and diag_file.exists():
        with np.load(cache_file) as saved:
            samples = {k: saved[k].copy() for k in ["alpha", "beta"]}
            if not (np.allclose(saved["center"], center) and np.allclose(saved["scale"], scale)):
                raise ValueError("Cached preprocessing mismatch.")
        diagnostic = json.loads(diag_file.read_text())
        diagnostic["cache_hit"] = True
    else:
        started = time.perf_counter()
        X_numpy = transform(train, center, scale)
        y_numpy = train.nominated.to_numpy(dtype=int)
        print(f"{predict_season} {category}: preparing MAP geometry ({len(train)} rows, {len(FEATURE_NAMES)} predictors)", flush=True)
        map_started = time.perf_counter()
        mode, covariance, map_gradient = initialize_geometry(X_numpy, y_numpy)
        map_seconds = time.perf_counter()-map_started
        print(f"MAP ready in {map_seconds:.1f}s; starting NUTS ({CHAIN_METHOD})", flush=True)
        X, y = jnp.asarray(X_numpy), jnp.asarray(y_numpy)
        kernel = NUTS(probit_model, target_accept_prob=TARGET_ACCEPT,
            max_tree_depth=MAX_TREE_DEPTH, dense_mass=True,
            inverse_mass_matrix=jnp.asarray(covariance), adapt_mass_matrix=False,
            init_strategy=init_to_value(values={"coef": jnp.asarray(mode)}))
        sampler = MCMC(kernel, num_warmup=NUM_WARMUP, num_samples=NUM_SAMPLES,
            num_chains=NUM_CHAINS, chain_method=CHAIN_METHOD, progress_bar=True)
        seed = SEED + 100*int(predict_season) + CATEGORIES.index(category)
        sampling_started = time.perf_counter()
        sampler.run(jax.random.PRNGKey(seed), X, y,
                    extra_fields=("diverging", "num_steps"))
        raw_samples = sampler.get_samples(group_by_chain=True)
        jax.block_until_ready(raw_samples)
        coef = np.asarray(raw_samples["coef"])
        samples = {"alpha": coef[..., 0], "beta": coef[..., 1:]}
        stats = mcmc_summary({"coef": coef}, prob=0.90, group_by_chain=True)["coef"]
        extras = sampler.get_extra_fields()
        steps = np.asarray(extras["num_steps"])
        diagnostic = dict(category=category, predict_season=int(predict_season),
            n_train=len(train), training_end=int(train.season.max()), n_predictors=len(FEATURE_NAMES),
            max_rhat=float(np.max(stats["r_hat"])), min_ess=float(np.min(stats["n_eff"])),
            divergences=int(np.asarray(extras["diverging"]).sum()),
            depth_limit_hits=int((steps >= 2**MAX_TREE_DEPTH-1).sum()),
            mean_num_steps=float(steps.mean()), median_num_steps=float(np.median(steps)),
            max_num_steps=int(steps.max()), map_seconds=map_seconds,
            map_max_gradient=map_gradient, mcmc_seconds=time.perf_counter()-sampling_started,
            seconds=time.perf_counter()-started, cache_hit=False)
        with cache_file.with_suffix(".tmp").open("wb") as stream:
            np.savez_compressed(stream, **samples, center=center, scale=scale,
                                posterior_mode=mode, metric_covariance=covariance)
        cache_file.with_suffix(".tmp").replace(cache_file)
        diag_file.write_text(json.dumps(diagnostic, indent=2))
        del sampler, kernel, X, y, raw_samples
        gc.collect()
    acceptable = (np.isfinite(diagnostic["max_rhat"]) and np.isfinite(diagnostic["min_ess"])
        and diagnostic["max_rhat"] <= MAX_RHAT and diagnostic["min_ess"] >= MIN_ESS
        and diagnostic["divergences"] == 0 and diagnostic["depth_limit_hits"] == 0)
    if not acceptable:
        raise RuntimeError(f"MCMC diagnostic failure: {diagnostic}. Inspect this fit before continuing; do not suppress the diagnostic gate.")
    return samples, center, scale, diagnostic

def posterior_probabilities(frame, samples, center, scale, chunk_size=256):
    alpha = samples["alpha"].reshape(-1)
    beta = samples["beta"].reshape(-1, len(FEATURE_NAMES))
    X = transform(frame, center, scale)
    mean, low, high = [], [], []
    for start in range(0, len(frame), chunk_size):
        p = ndtr(alpha[:, None] + beta @ X[start:start+chunk_size].T)
        mean.extend(p.mean(axis=0))
        interval = np.quantile(p, [0.05, 0.95], axis=0)
        low.extend(interval[0]); high.extend(interval[1])
    return np.asarray(mean), np.asarray(low), np.asarray(high)

## 5a. One-fit pilot before the backtest

Run through this cell first. It fits the first planned category-season with the actual 300-warmup/500-draw settings and caches it. Inspect total time, R-hat, ESS, divergences, tree-depth hits and mean trajectory steps. The earlier short benchmark's `1023` step cap is not a desired operating point. A rough total-time projection is printed, but other categories can differ substantially. Starting the backtest afterwards reuses this completed fit.

If the diagnostic gate raises an error, stop and inspect that fit rather than starting the full backtest.

In [ ]:
pilot_season = TEST_SEASONS[0]
pilot_cat = CATEGORIES[0]
pilot_train = historical.loc[(historical.season < pilot_season) & (historical.cat == pilot_cat)].copy()
print("Devices:", jax.devices(), "| method:", CHAIN_METHOD, flush=True)
pilot_samples, pilot_center, pilot_scale, pilot_diagnostic = fit_probit(pilot_train, pilot_cat, pilot_season)
display(pd.Series(pilot_diagnostic))
print("Very rough total-fit projection (minutes):",
      round(pilot_diagnostic["seconds"]*len(CATEGORIES)*(len(TEST_SEASONS)+1)/60, 1))
print("Projection assumes every fit has this pilot's time; it is not a runtime guarantee.")
del pilot_samples
gc.collect()

## 6. Rolling-origin backtest and scores

Fit each category separately for seasons 2022–2025, always using earlier seasons only. The equal-probability baseline remains the historical category nomination rate.

Brier is the mean squared error of the artist-level probabilities. Precision@5 is the observed nomination fraction among the five highest-probability artists. For the non-ranking baseline, report the exact expected precision of a uniformly random five-artist list, equal to the held-out nomination rate. Test labels enter scoring only.

Scores give equal weight to category-season events. The compact feature map and four test seasons are fixed in advance of this revision's scores. All eight target categories are retained.

In [ ]:
def score_predictions(predictions):
    rows = []
    for (season, cat), block in predictions.groupby(["season", "cat"], sort=True):
        y = block.nominated.to_numpy()
        if len(block) < TOP_K:
            raise ValueError("Insufficient candidates for a top-five shortlist.")
        top = block.sort_values(["p_probit", "artist_key"], ascending=[False, True]).head(TOP_K)
        for model, column, precision in [
            ("baseline", "p_baseline", y.mean()),
            ("probit", "p_probit", top.nominated.mean())]:
            rows.append(dict(season=int(season), cat=cat, model=model,
                n_candidates=len(block), actual_nominated=int(y.sum()),
                expected_nominated=float(block[column].sum()),
                brier=float(np.mean((block[column].to_numpy()-y)**2)),
                precision_at_5=float(precision)))
    return pd.DataFrame(rows)

backtest_parts, diagnostics = [], []
for test_season in TEST_SEASONS:
    for cat in CATEGORIES:
        train = historical.loc[(historical.season < test_season) & (historical.cat == cat)].copy()
        test = historical.loc[(historical.season == test_season) & (historical.cat == cat)].copy()
        if test.empty:
            raise ValueError(f"Missing evaluation event: {test_season}, {cat}")
        print(f"Backtest event: {test_season}, {cat}", flush=True)
        samples, center, scale, diagnostic = fit_probit(train, cat, test_season)
        p, lo, hi = posterior_probabilities(test, samples, center, scale)
        result = test[["season", "cat", "artist_key", "artist", "nominated"]].copy()
        result["p_baseline"] = float(train.nominated.mean())
        result["p_probit"], result["p_probit_q05"], result["p_probit_q95"] = p, lo, hi
        backtest_parts.append(result); diagnostics.append(diagnostic)
        # Checkpoints allow recovery and preserve every completed event.
        pd.concat(backtest_parts).to_csv(OUTPUT / "backtest_predictions.csv.gz", index=False)
        pd.DataFrame(diagnostics).to_csv(OUTPUT / "mcmc_diagnostics.csv", index=False)
        print(f"{test_season} {cat}: Rhat={diagnostic['max_rhat']:.3f}, "
              f"ESS={diagnostic['min_ess']:.0f}, mean steps={diagnostic['mean_num_steps']:.1f}, {diagnostic['seconds']:.1f}s "
              f"{'(cached)' if diagnostic['cache_hit'] else ''}", flush=True)
        del samples
    jax.clear_caches()
    gc.collect()

backtest = pd.concat(backtest_parts, ignore_index=True)
prob_cols = ["p_baseline", "p_probit", "p_probit_q05", "p_probit_q95"]
if not np.isfinite(backtest[prob_cols]).all().all() or not backtest[prob_cols].ge(0).all().all() or not backtest[prob_cols].le(1).all().all():
    raise ValueError("Invalid predicted probabilities.")
if backtest.duplicated(keys).any():
    raise ValueError("Duplicate held-out predictions.")
scores = score_predictions(backtest)
assert len(scores) == len(TEST_SEASONS) * len(CATEGORIES) * 2
scores.to_csv(OUTPUT / "scores_by_category_season.csv", index=False)
overall = scores.groupby("model")[["brier", "precision_at_5"]].mean()
display(overall)
display(scores.groupby(["cat", "model"])[["brier", "precision_at_5"]].mean())

## 7. Paired model comparison

Artists and categories in a ceremony are dependent. Therefore, the comparison first averages each model's scores over the eight categories **within a season**, then forms one paired difference per season. Positive improvement means the probit is better: baseline minus probit for Brier, probit minus baseline for precision.

We report a two-sided paired t test and a 95% t interval on these seasonal differences. This is an **exploratory small-sample comparison**, not a Diebold–Mariano test with HAC adjustment: there are only four seasons, training windows overlap, and season-to-season independence is not guaranteed. The four-season comparison has very limited statistical power. Thousands of artist rows are not treated as independent forecast occasions. Brier is the primary comparison; precision is secondary, without a multiple-testing claim.

In [ ]:
yearly = scores.groupby(["season", "model"])[["brier", "precision_at_5"]].mean().unstack("model")
differences = pd.DataFrame({
    "brier_improvement": yearly["brier"]["baseline"] - yearly["brier"]["probit"],
    "precision_improvement": yearly["precision_at_5"]["probit"] - yearly["precision_at_5"]["baseline"],
})

def paired_summary(values):
    x = np.asarray(values, dtype=float)
    n, mean = len(x), float(x.mean())
    if n < 2:
        return dict(n_seasons=n, mean_improvement=mean, ci95_low=np.nan, ci95_high=np.nan,
                    t_stat=np.nan, p_two_sided=np.nan)
    se = x.std(ddof=1) / np.sqrt(n)
    if se == 0:
        tstat = 0.0 if mean == 0 else np.sign(mean) * np.inf
        pvalue = 1.0 if mean == 0 else 0.0
    else:
        tstat = mean / se
        pvalue = float(2 * student_t.sf(abs(tstat), df=n-1))
    margin = float(student_t.ppf(0.975, df=n-1) * se)
    return dict(n_seasons=n, mean_improvement=mean, ci95_low=mean-margin,
                ci95_high=mean+margin, t_stat=tstat, p_two_sided=pvalue)

comparison = pd.DataFrame({name: paired_summary(differences[name]) for name in differences}).T
display(differences)
display(comparison)
differences.to_csv(OUTPUT / "paired_season_differences.csv")
comparison.to_csv(OUTPUT / "paired_model_comparison.csv")

## 8. Calibration and year-by-year performance

The reliability table compares average forecasts with nomination frequency in prespecified probability bins. It pools held-out observations and is descriptive: observations within an artist or season are dependent, and high-probability bins can be small. It is not used to recalibrate or select the model. The figure also shows whether the Brier advantage is consistent across seasons.

In [ ]:
bins = np.array([0, 0.001, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 1.0])
calibration_parts = []
for model, col in [("baseline", "p_baseline"), ("probit", "p_probit")]:
    b = backtest.assign(probability=backtest[col])
    b["bin"] = pd.cut(b.probability, bins=bins, include_lowest=True)
    table = b.groupby("bin", observed=True).agg(
        n=("nominated", "size"), mean_probability=("probability", "mean"),
        nomination_frequency=("nominated", "mean")).reset_index()
    table["model"] = model
    calibration_parts.append(table)
calibration = pd.concat(calibration_parts, ignore_index=True)
calibration.to_csv(OUTPUT / "calibration.csv", index=False)
display(calibration)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for model in ["baseline", "probit"]:
    axes[0].plot(yearly.index, yearly["brier"][model], marker="o", label=model)
    c = calibration.loc[calibration.model == model]
    axes[1].plot(c.mean_probability, c.nomination_frequency, marker="o", label=model)
axes[0].set(xlabel="Held-out season", ylabel="Mean category Brier score", title="Historical forecast error")
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].set(xlabel="Mean predicted probability", ylabel="Observed nomination frequency",
            title="Held-out calibration", xlim=(0, 1), ylim=(0, 1))
for ax in axes:
    ax.legend(); ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT / "backtest_and_calibration.png", dpi=160)
plt.show()

## 9. Final fit and artist shortlists for the 2027 ceremony

Each category is fitted on all labelled seasons through 2025. We retain the top five artists by posterior mean probability. The 5th and 95th percentiles describe uncertainty about an artist's **nomination probability**, not a 90% interval for the binary nomination outcome and not joint uncertainty about the shortlist. Predictions for all candidates are exported, including the equal-probability baseline.

Probabilities are not normalized across artists. Several artists can be nominated together, and more or fewer than five artists may be nominated. An artist-level shortlist cannot be interpreted as five distinct eligible works.

In [ ]:
forecast_parts, coefficient_parts = [], []
for cat in CATEGORIES:
    train = historical.loc[historical.cat == cat].copy()
    test = future.loc[future.cat == cat].copy()
    samples, center, scale, diagnostic = fit_probit(train, cat, FORECAST_SEASON)
    p, lo, hi = posterior_probabilities(test, samples, center, scale)
    result = test[["season", "cat", "artist_key", "artist"]].copy()
    result["p_baseline"] = float(train.nominated.mean())
    result["p_probit"], result["p_probit_q05"], result["p_probit_q95"] = p, lo, hi
    forecast_parts.append(result); diagnostics.append(diagnostic)
    beta = samples["beta"].reshape(-1, len(FEATURE_NAMES))
    coefficient_parts.append(pd.DataFrame({"cat": cat, "feature": FEATURE_NAMES,
        "posterior_mean": beta.mean(axis=0), "q05": np.quantile(beta, 0.05, axis=0),
        "q95": np.quantile(beta, 0.95, axis=0)}))
    print(f"Final {cat}: Rhat={diagnostic['max_rhat']:.3f}, ESS={diagnostic['min_ess']:.0f}", flush=True)
    del samples

forecast = pd.concat(forecast_parts, ignore_index=True)
forecast = forecast.sort_values(["cat", "p_probit", "artist_key"], ascending=[True, False, True])
forecast["rank_in_category"] = forecast.groupby("cat").cumcount() + 1
shortlists = forecast.loc[forecast.rank_in_category <= TOP_K].copy()
assert len(shortlists) == TOP_K * len(CATEGORIES)
assert not forecast.duplicated(keys).any()
assert np.isfinite(forecast[prob_cols]).all().all()
assert forecast[prob_cols].ge(0).all().all() and forecast[prob_cols].le(1).all().all()
coefficients = pd.concat(coefficient_parts, ignore_index=True)
forecast.to_csv(OUTPUT / "forecast_2027_all_artists.csv", index=False)
shortlists.to_csv(OUTPUT / "forecast_2027_top5.csv", index=False)
coefficients.to_csv(OUTPUT / "final_standardized_coefficients.csv", index=False)
diagnostics_df = pd.DataFrame(diagnostics)
diagnostics_df.to_csv(OUTPUT / "mcmc_diagnostics.csv", index=False)
display(shortlists[["cat", "rank_in_category", "artist", "p_probit", "p_probit_q05", "p_probit_q95"]])
display(diagnostics_df[["category", "predict_season", "max_rhat", "min_ess", "divergences", "depth_limit_hits"]])

## 10. Computed conclusions and reproducibility record

The following summary is generated from the computed scores. It must not be filled manually with anticipated results. Coefficients in the exported table are conditional associations on the latent probit scale per training standard deviation, not causal effects or percentage-point effects.

In [ ]:
b0, b1 = overall.loc["baseline", "brier"], overall.loc["probit", "brier"]
print(f"Backtest: {min(TEST_SEASONS)}–{max(TEST_SEASONS)}, {len(TEST_SEASONS)} seasons, "
      f"{len(TEST_SEASONS)*len(CATEGORIES)} category-season events.")
print(f"Brier: baseline={b0:.6f}; probit={b1:.6f}; relative reduction={(b0-b1)/b0:.1%}.")
print(f"Precision@5: random-baseline expectation={overall.loc['baseline', 'precision_at_5']:.3f}; "
      f"probit={overall.loc['probit', 'precision_at_5']:.3f}.")
row = comparison.loc["brier_improvement"]
print(f"Season-level mean Brier improvement={row.mean_improvement:.6f}; "
      f"95% t interval=[{row.ci95_low:.6f}, {row.ci95_high:.6f}]; "
      f"exploratory two-sided p={row.p_two_sided:.4g}.")
print("Interpret the test cautiously: few seasons, dependent categories and overlapping training windows.")
record = dict(created_at_utc=datetime.now(timezone.utc).isoformat(), data_sha256=DATA_SHA256,
    model_version=MODEL_VERSION, scope="artist-level nominations only", forecast_season=FORECAST_SEASON,
    chart_cutoff="2026-09-30 (proxy window in supplied dataset)",
    test_seasons=TEST_SEASONS, feature_names=FEATURE_NAMES, seed=SEED,
    award_input_columns=AWARD_COLUMNS, relevant_award_mapping=RELEVANT_AWARD_STEMS,
    preconditioner=PRECONDITIONER, chain_method=CHAIN_METHOD,
    award_alignment="Supplied season mapping: current nominations for all three awards; current VMA/iHeart wins; most recent earlier-season AMA wins; no additional shift.",
    num_warmup=NUM_WARMUP, samples_per_chain=NUM_SAMPLES, chains=NUM_CHAINS,
    target_accept=TARGET_ACCEPT, max_tree_depth=MAX_TREE_DEPTH,
    prior=dict(alpha_location=ALPHA_LOCATION, alpha_scale=ALPHA_SCALE, beta_scale=BETA_SCALE),
    versions=VERSIONS, completed_fits=len(diagnostics_df),
    max_rhat=float(diagnostics_df.max_rhat.max()), min_ess=float(diagnostics_df.min_ess.min()))
(OUTPUT / "run_metadata.json").write_text(json.dumps(record, indent=2))
print("Posterior mean leapfrog steps across fits:", diagnostics_df.mean_num_steps.mean())
print("Completed fits:", len(diagnostics_df), "| saved results:", OUTPUT.resolve())

## 11. Interpretation limits and course-requirement check

| Requirement | What this notebook supplies |
|---|---|
| Target, forecast form and horizon | Artist-level nomination probabilities and top-five rankings for the upcoming ceremony |
| Baseline and extension | Historical equal probability versus category-specific Bayesian probit |
| Several forecast occasions | Four chronological test seasons; no random row split |
| Appropriate scores | Brier score and precision@5, matching Stage 1 of the proposal |
| Model comparison | Paired seasonal differences, uncertainty intervals and exploratory t tests |
| Bayesian computation | Explicit priors, NUTS, posterior-average probabilities, diagnostics |
| Reproducibility | Included input snapshot, hash and seeds; versions, predictions and posterior cache are saved when run |
| Full project proposal | **Not complete:** winner probabilities and award-count distributions remain to be implemented |

**Remaining data limitations.** The fixed October–September chart window approximates rather than reproduces each ceremony's eligibility dates. The upstream artist-credit splitter uses a dictionary constructed from the full chart history; historical entity resolution therefore is not a fully time-frozen reconstruction. Supplemental labels were assembled retrospectively and have not all been independently audited. All model features and transformations here obey the season split, but the exercise should be described as a **historical backtest on a retrospectively assembled dataset**, not as a perfectly reconstructed real-time information set.

The 27-feature aggregation is a deliberate specification change. Relevant-category mappings are proxies and can discard useful fine detail. The fixed local MAP metric can be inadequate if the posterior is strongly non-Gaussian, so diagnostics and posterior checks remain necessary. The shorter four-season evaluation limits evidence about statistical significance.

Other-award features follow the supplied seasonal alignment. A year label alone does not prove that every event was published before a historical forecast date; any subsequently discovered calendar exception should be corrected upstream and then propagated through a fresh backtest. No performance numbers from the earlier charts/Grammy-only model establish the quality of this expanded model.

The artist pool omits non-charting nominees; scores concern the supplied pool and do not measure performance on every possible nominee. The model does not enforce eligibility or the number of nomination slots. It merges historical predecessor categories for nomination modelling, as in the source table. Calendar changes, incomplete early artist histories, changing candidate-pool size and changes in Grammy voting may reduce transferability to 2027.

The likelihood assumes conditional independence of artist-season outcomes within a category. Shared works and repeated artists can leave residual dependence, so the posterior intervals are model-conditional and may understate uncertainty. Averaging score differences by season avoids treating individual artist rows as independent in the model-comparison test, but does not remove this modelling limitation.

**AI-use disclosure.** ChatGPT was used to inspect the supplied materials, draft and check the modelling notebook, and explain modelling choices. The team remains responsible for validating data sources, understanding the methods and reviewing all submitted results.